In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import pandas as pd
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/tltommu/distributed-signal-coherence-classification/sample_submission.csv
/kaggle/input/datasets/tltommu/distributed-signal-coherence-classification/train.csv
/kaggle/input/datasets/tltommu/distributed-signal-coherence-classification/test.csv


In [2]:
test = pd.read_csv('/kaggle/input/datasets/tltommu/distributed-signal-coherence-classification/test.csv')
train = pd.read_csv('/kaggle/input/datasets/tltommu/distributed-signal-coherence-classification/train.csv')

In [32]:

RANDOM_STATE = 42
N_SPLITS = 7
TARGET = "signal_incoherence_flag"
ID_COL = "sample_id"
def optimize_threshold(y_true, y_probs):

    thresholds = np.linspace(0.01, 0.20, 200)

    best_mcc = -1
    best_threshold = 0.2

    for t in thresholds:
        preds = (y_probs > t).astype(int)
        score = matthews_corrcoef(y_true, preds)

        if score > best_mcc:
            best_mcc = score
            best_threshold = t

    return best_threshold, best_mcc

# -----------------------------
# LightGBM training function
# -----------------------------
def train_lgbm(X, y, X_test, n_splits=7, random_state=42):
    oof_preds = np.zeros(X.shape[0])
    test_preds = np.zeros(X_test.shape[0])
    positive_rate = y.mean()

    k = int(len(test_preds) * positive_rate)

    top_idx = np.argsort(test_preds)[-k:]

    final_test_preds = np.zeros(len(test_preds))
    final_test_preds[top_idx] = 1
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"========== LGBM Fold {fold} ==========")
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(
            n_estimators=3000,
    learning_rate=0.02,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=30,
    scale_pos_weight=20,
    random_state=random_state,
    verbosity=-1
        )
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_metric='binary_logloss',
            callbacks=[lgb.early_stopping(100)]
        )
        
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / n_splits
    
    return oof_preds, test_preds

# -----------------------------
# Full pipeline
# -----------------------------
def run_pipeline(model_type="lgbm"):
    """
    model_type: "lgbm" only
    This version includes noise features and handles class imbalance.
    Outputs are pushed to globals for convenience.
    """
    print("Loading data...")
    train = pd.read_csv('/kaggle/input/datasets/tltommu/distributed-signal-coherence-classification/train.csv')
    test = pd.read_csv('/kaggle/input/datasets/tltommu/distributed-signal-coherence-classification/test.csv')
    
    train.columns = train.columns.str.strip()
    test.columns = test.columns.str.strip()
    
    # Include all features except ID and TARGET
    features = [col for col in train.columns if col not in [ID_COL, TARGET]]
    
    X = train[features]
    y = train[TARGET]
    X_test = test[features]
    
    print("Using", len(features), "features (including noise)")
    
    # Train LightGBM
    oof_preds, test_preds = train_lgbm(X, y, X_test)
    
    # OOF statistics
    print("OOF min:", oof_preds.min())
    print("OOF max:", oof_preds.max())
    print("OOF mean:", oof_preds.mean())
    print("Positive rate:", y.mean())
    
    # Optimize MCC threshold
    best_threshold, best_mcc = optimize_threshold(y, oof_preds)
    final_score = (best_mcc + 1) / 2
    
    print("\n====================================")
    print(f"Best MCC: {best_mcc:.5f}")
    print(f"Final Score (Scaled MCC): {final_score:.5f}")
    print(f"Optimal Threshold: {best_threshold:.2f}")
    print("====================================")
    
    # Push to globals
    globals().update({
        "oof_preds": oof_preds,
        "test_preds": test_preds,
        "best_threshold": best_threshold,
        "best_mcc": best_mcc,
        "y":y
    })
    
    return oof_preds, test_preds, best_threshold, best_mcc,y

In [33]:
run_pipeline('lgbm')
positive_rate = y.mean()

k = int(len(test_preds) * positive_rate)

top_idx = np.argsort(test_preds)[-k:]

final_test_preds = np.zeros(len(test_preds))
final_test_preds[top_idx] = 1

Loading data...
Using 45 features (including noise)
========== LGBM Fold 1 ==========
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[18]	valid_0's binary_logloss: 0.145126
========== LGBM Fold 2 ==========
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1]	valid_0's binary_logloss: 0.134141
========== LGBM Fold 3 ==========
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[2]	valid_0's binary_logloss: 0.128185
========== LGBM Fold 4 ==========
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1]	valid_0's binary_logloss: 0.12911
========== LGBM Fold 5 ==========
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[14]	valid_0's binary_logloss: 0.122452
========== LGBM Fold 6 ==========
Training until validation scores don't improve for 100 rounds
E

In [34]:
final_test_preds = (test_preds > best_threshold).astype(int)

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: final_test_preds
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission saved to /kaggle/working/submission.csv")

Submission saved to /kaggle/working/submission.csv
